# Tutorial 02: Anomaly Detection on Industrial Sensor Data (SWaT)

This notebook demonstrates unsupervised anomaly detection on industrial sensor streams
using an **LSTM Autoencoder**. The approach is widely used in IEEE IES and SCADA security research.

**Key concepts covered:**
- Reconstruction-based anomaly detection
- Best-threshold search (F1 maximisation)
- **Point-Adjust (PA)** evaluation — the standard metric for ICS anomaly benchmarks
- ROC-AUC, F1, Precision, Recall reporting

**Dataset:** Synthetic SWaT-like data (51 sensors, 5% anomaly segments). No download required.

**Reference:** Mathur & Tippenhauer, *SWaT: A Water Treatment Testbed for Research and Training on ICS Security*, IEEE IES 2016.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt

from datasets.dataloader import get_dummy_swat_dataloader
from models.lstm_forecasting import LSTMAutoencoder
from anomaly_detection.pipeline import ReconstructionAnomalyPipeline
from visualization.plot_utils import plot_anomaly_scores

## 1. Load Data

The synthetic SWaT loader generates 51-channel data with **contiguous anomaly segments**
(more realistic than isolated anomaly points) injected as distribution shifts of +3σ.

In [ ]:
NUM_FEATURES = 51
WINDOW_SIZE  = 100
BATCH_SIZE   = 64

train_loader = get_dummy_swat_dataloader(
    batch_size=BATCH_SIZE,
    window_size=WINDOW_SIZE,
    num_samples=8000,
    num_features=NUM_FEATURES,
)

test_loader = get_dummy_swat_dataloader(
    batch_size=BATCH_SIZE,
    window_size=WINDOW_SIZE,
    num_samples=2000,
    num_features=NUM_FEATURES,
)

print(f"Train batches: {len(train_loader)}  |  Test batches: {len(test_loader)}")
print(f"Window shape per sample: ({WINDOW_SIZE}, {NUM_FEATURES})")

## 2. Build LSTM Autoencoder

Architecture:
- **Encoder**: 2-layer LSTM compresses `(B, 100, 51)` → hidden state `(B, 64)`
- **Decoder**: LSTM reconstructs the sequence from the hidden state
- **Loss**: MSE between input and reconstruction
- **Anomaly score**: mean reconstruction error per window

In [ ]:
model = LSTMAutoencoder(
    num_features=NUM_FEATURES,
    hidden_dim=64,
    num_layers=2,
)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"LSTMAutoencoder parameters: {n_params:,}")

## 3. Train

The pipeline trains on mixed data (normal + anomalous windows). Because anomalies are rare,
the model learns to reconstruct normal patterns well — anomalies produce high reconstruction error.

In [ ]:
pipeline = ReconstructionAnomalyPipeline(
    model=model,
    learning_rate=1e-3,
    model_name="LSTMAutoencoder",
    dataset_name="SWaT_dummy",
)

metrics = pipeline.fit(
    train_loader,
    test_loader=test_loader,
    epochs=8,
)

## 4. Visualise Anomaly Scores

We collect per-sample reconstruction errors and overlay them with the ground truth labels.

In [ ]:
scores, labels = pipeline.compute_anomaly_scores(test_loader)
threshold = metrics.best_threshold

print(f"Anomaly score range: [{scores.min():.4f}, {scores.max():.4f}]")
print(f"Best threshold (max F1): {threshold:.4f}")
print(f"Anomaly ratio in test:   {labels.mean():.2%}")

# Build a representative signal array for visualisation
# (we show a straight reconstruction-error timeline here)
signal_proxy = np.stack([scores] * 5, axis=1)  # fake 5-channel for plotting

fig = plot_anomaly_scores(
    true_data=signal_proxy,
    anomaly_scores=scores,
    labels=labels,
    threshold=threshold,
    feature_idx=0,
    title="LSTMAutoencoder — Reconstruction Error on SWaT (dummy)",
)
plt.show()

## 5. Results Summary

The Point-Adjust (PA) metric credits entire anomaly segments when any point within the segment
is detected. This is the standard evaluation used in AnomalyTransformer, TranAD, and most
ICS anomaly detection papers.

In [ ]:
print("=" * 40)
print("  Anomaly Detection Results")
print("=" * 40)
print(f"  ROC-AUC       {metrics.roc_auc:.4f}")
print(f"  F1            {metrics.f1:.4f}")
print(f"  F1 (PA)       {metrics.f1_pa:.4f}  ← point-adjust")
print(f"  Precision     {metrics.precision:.4f}")
print(f"  Recall        {metrics.recall:.4f}")
print(f"  Threshold     {metrics.best_threshold:.6f}")